In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from processing import *
from fit_pv import *
from wavelengths import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

In [4]:
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'

In [15]:
folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2025-09-16/'
#folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2024-07-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))

data, header = process(files[2], dark_file=dark_file, prefilter_file=prefilter_file)
data = np.squeeze(data)

binning = 8
data = rebin(data, binning)
wv = read_wavelengths(header)

In [16]:
params = fit_pv(-np.moveaxis(data, 0, -1), wv)
shift = params[...,0]
fwhm = params[...,1]
depth = params[...,2]
offset = -params[...,3]

In [17]:
mask = data[0] > 1000

plt.figure(figsize=(12,10))
plt.imshow(fwhm * mask, 'inferno', vmin=0.15, vmax=0.2)
plt.colorbar(label=r'Line width, $\AA$')
plt.tight_layout()

In [18]:
mask = data[0] > 1000

plt.figure(figsize=(12,10))
plt.imshow(depth / offset * mask, 'inferno', vmin=0.05, vmax=0.1)
plt.colorbar(label='Line depth')
plt.tight_layout()